# Comparison Across Training Seeds

This notebook compares final-evaluation results across training seeds `156`, `157`, and `158` for each task, using experiments `001–003`.

The related unseen split (`test_unseen_related`) is excluded for object and object-region classification because these results are not scored. `validation_unseen_matched` is included for reference and is not an independent test result.

It exports one CSV summary table per available task, containing individual seed values and summary statistics. Tasks without an `experiments.parquet` file are skipped.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

In [2]:
RESULTS_DIR = Path("../results/final_evaluation")
OUTPUT_DIR = Path("../exports/seed_comparison")
EXPERIMENT_NUMBERS = [1, 2, 3]
SEEDS = [156, 157, 158]

EVALUATION_SPLITS = [
    "test",
    "test_unseen_matched",
    "test_unseen_related",
    "validation_unseen_matched",
]

TASK_METRICS = {
    "object_classification": ["accuracy", "macro_f1_avg"],
    "object_region_classification": ["accuracy", "macro_f1_avg"],
    "force_level_classification": ["accuracy", "macro_f1_avg"],
    "object_conditioned_classification": ["accuracy", "macro_f1_avg"],
    "object_region_conditioned_classification": ["accuracy", "macro_f1_avg"],
    "force_n_regression": ["mae", "rmse", "r2"],
    "fsr_voltage_regression": ["mae", "rmse", "r2"],
}

UNSCORED_RELATED_TASKS = {
    "object_classification",
    "object_region_classification",
    "object_conditioned_classification",
    "object_region_conditioned_classification",
}

In [3]:
def load_seed_results(task):
    """
    Load one final-evaluation run per experiment and training seed.

    Args:
        task (str): The task folder under the final-evaluation results directory.

    Returns:
        pd.DataFrame: The selected result rows indexed in training-seed order.
    """

    results_path = RESULTS_DIR / task / "experiments.parquet"
    results = pd.read_parquet(results_path)
    results = results.loc[results["experiment_number"].isin(EXPERIMENT_NUMBERS)].copy()

    # Keep seed values and summary columns in the same order across all task tables
    return results.set_index("seed").loc[SEEDS]


def calculate_seed_summary(task, results):
    """
    Calculate per-split metric statistics across the three training seeds.

    Args:
        task (str): The task folder identifying the metrics and scored splits.
        results (pd.DataFrame): The selected result rows indexed by training seed.

    Returns:
        pd.DataFrame: Individual seed values and summary statistics per metric.
    """

    rows = []
    for split in EVALUATION_SPLITS:
        # Skip unscored classification results while retaining force-task metrics
        if split == "test_unseen_related" and task in UNSCORED_RELATED_TASKS:
            continue

        for metric in TASK_METRICS[task]:
            column = f"{split}_{metric}"
            values = results.loc[SEEDS, column]
            row = {"split": split, "metric": metric}
            row.update({f"seed_{seed}": values.loc[seed] for seed in SEEDS})
            row.update(
                {
                    "n": values.count(),
                    "mean": values.mean(),
                    "std": values.std(ddof=1),
                    "min": values.min(),
                    "max": values.max(),
                }
            )
            rows.append(row)

    return pd.DataFrame(rows)


def export_seed_comparisons():
    """
    Export a seed-comparison CSV table for each available final-evaluation task.

    Tasks without an experiments.parquet file are skipped.

    Returns:
        dict[str, pd.DataFrame]: The summary table for each task.
    """

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    comparisons = {}
    for task in TASK_METRICS:
        if not (RESULTS_DIR / task / "experiments.parquet").is_file():
            print(f"Skipping {task}: experiments.parquet not found.")
            continue

        results = load_seed_results(task)
        comparison = calculate_seed_summary(task, results)
        comparison.to_csv(OUTPUT_DIR / f"{task}.csv", index=False)
        comparisons[task] = comparison

    return comparisons

In [4]:
# Run available task comparisons and retain the tables for display below
seed_comparisons = export_seed_comparisons()

Skipping object_conditioned_classification: experiments.parquet not found.
Skipping object_region_conditioned_classification: experiments.parquet not found.


In [5]:
# Display each task table
with pd.option_context("display.max_rows", None, "display.precision", 4):
    for task, comparison in seed_comparisons.items():
        display(task)
        display(comparison)

'object_classification'

,split,metric,seed_156,seed_157,seed_158,n,mean,std,min,max
0,test,accuracy,92.1327,91.3377,93.3498,3,92.2734,1.0134,91.3377,93.3498
1,test,macro_f1_avg,0.9233,0.9149,0.9346,3,0.9243,0.0098,0.9149,0.9346
2,test_unseen_matched,accuracy,47.7469,50.4321,52.1296,3,50.1029,2.2098,47.7469,52.1296
3,test_unseen_matched,macro_f1_avg,0.5990,0.5978,0.6297,3,0.6088,0.0181,0.5978,0.6297
4,validation_unseen_matched,accuracy,53.8194,50.2639,54.0000,3,52.6944,2.1069,50.2639,54.0000
5,validation_unseen_matched,macro_f1_avg,0.5757,0.5489,0.5771,3,0.5672,0.0159,0.5489,0.5771


'object_region_classification'

,split,metric,seed_156,seed_157,seed_158,n,mean,std,min,max
0,test,accuracy,88.3662,88.4375,88.4759,3,88.4265,0.0556,88.3662,88.4759
1,test,macro_f1_avg,0.8752,0.8740,0.8767,3,0.8753,0.0014,0.8740,0.8767
2,test_unseen_matched,accuracy,34.4753,36.2500,35.5093,3,35.4115,0.8914,34.4753,36.2500
3,test_unseen_matched,macro_f1_avg,0.4220,0.4547,0.4338,3,0.4369,0.0166,0.4220,0.4547
4,validation_unseen_matched,accuracy,42.2083,41.8750,40.0000,3,41.3611,1.1905,40.0000,42.2083
5,validation_unseen_matched,macro_f1_avg,0.4622,0.4655,0.4328,3,0.4535,0.0180,0.4328,0.4655


'force_level_classification'

,split,metric,seed_156,seed_157,seed_158,n,mean,std,min,max
0,test,accuracy,88.4430,89.9068,88.7390,3,89.0296,0.7740,88.4430,89.9068
1,test,macro_f1_avg,0.8768,0.8903,0.8776,3,0.8816,0.0076,0.8768,0.8903
2,test_unseen_matched,accuracy,64.9537,69.4907,73.9506,3,69.4650,4.4985,64.9537,73.9506
3,test_unseen_matched,macro_f1_avg,0.6371,0.6875,0.7361,3,0.6869,0.0495,0.6371,0.7361
4,test_unseen_related,accuracy,74.4097,76.7535,75.3993,3,75.5208,1.1766,74.4097,76.7535
5,test_unseen_related,macro_f1_avg,0.7468,0.7683,0.7516,3,0.7556,0.0113,0.7468,0.7683
6,validation_unseen_matched,accuracy,73.0417,75.5972,77.5417,3,75.3935,2.2569,73.0417,77.5417
7,validation_unseen_matched,macro_f1_avg,0.7290,0.7555,0.7757,3,0.7534,0.0235,0.7290,0.7757


'force_n_regression'

,split,metric,seed_156,seed_157,seed_158,n,mean,std,min,max
0,test,mae,1.2678,1.2557,1.3041,3,1.2759,0.0252,1.2557,1.3041
1,test,rmse,2.0169,1.9382,1.9589,3,1.9713,0.0408,1.9382,2.0169
2,test,r2,0.7635,0.7816,0.7769,3,0.7740,0.0094,0.7635,0.7816
3,test_unseen_matched,mae,1.5272,1.6739,1.6836,3,1.6282,0.0876,1.5272,1.6836
4,test_unseen_matched,rmse,2.2189,2.3228,2.2887,3,2.2768,0.0530,2.2189,2.3228
5,test_unseen_matched,r2,0.6807,0.6501,0.6603,3,0.6637,0.0156,0.6501,0.6807
6,test_unseen_related,mae,1.4814,1.4877,1.5152,3,1.4948,0.0180,1.4814,1.5152
7,test_unseen_related,rmse,2.2769,2.2373,2.2511,3,2.2551,0.0201,2.2373,2.2769
8,test_unseen_related,r2,0.6570,0.6689,0.6648,3,0.6636,0.0060,0.6570,0.6689
9,validation_unseen_matched,mae,1.3685,1.4795,1.4533,3,1.4338,0.0580,1.3685,1.4795


'fsr_voltage_regression'

,split,metric,seed_156,seed_157,seed_158,n,mean,std,min,max
0,test,mae,0.3317,0.3322,0.3121,3,0.3253,0.0115,0.3121,0.3322
1,test,rmse,0.6336,0.6087,0.5678,3,0.6033,0.0332,0.5678,0.6336
2,test,r2,0.7889,0.8052,0.8305,3,0.8082,0.0209,0.7889,0.8305
3,test_unseen_matched,mae,0.4508,0.5021,0.5446,3,0.4992,0.0470,0.4508,0.5446
4,test_unseen_matched,rmse,0.8509,0.8829,0.9614,3,0.8984,0.0569,0.8509,0.9614
5,test_unseen_matched,r2,0.6278,0.5993,0.5249,3,0.5840,0.0532,0.5249,0.6278
6,test_unseen_related,mae,0.3900,0.4203,0.4722,3,0.4275,0.0416,0.3900,0.4722
7,test_unseen_related,rmse,0.7242,0.7322,0.8378,3,0.7647,0.0634,0.7242,0.8378
8,test_unseen_related,r2,0.7278,0.7217,0.6357,3,0.6950,0.0515,0.6357,0.7278
9,validation_unseen_matched,mae,0.4157,0.4783,0.4790,3,0.4577,0.0364,0.4157,0.4790
